# Traffic data exploration

Set `DATA_PATH` to a locally downloaded CSV or HDF5 dataset. This notebook does not generate example traffic values. METR-LA is primarily a speed dataset, so only plot available observed metrics.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_preprocessing import load_traffic_data, regularize_series
DATA_PATH = ROOT / 'datasets' / 'traffic.csv'  # Change this to your downloaded dataset
TARGET = 'speed'
raw, quality, coordinates = load_traffic_data(DATA_PATH, target=TARGET)
data = regularize_series(raw, interval_minutes=5)
quality, data.shape

In [ ]:
data.describe().T.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
data.stack().plot.hist(bins=40, ax=axes[0], title=f'{TARGET} distribution')
hourly = data.groupby(data.index.hour).mean().mean(axis=1)
hourly.plot(ax=axes[1], title=f'Mean {TARGET} by hour', xlabel='Hour of day')
plt.tight_layout()

In [ ]:
sample_sensors = list(data.columns[:min(12, data.shape[1])])
sns.heatmap(data[sample_sensors].corr(), cmap='coolwarm', center=0)
plt.title('Sensor correlation (illustrative subset)')
plt.tight_layout()

In [ ]:
sensor = data.columns[0]
data[sensor].plot(figsize=(14, 4), title=f'Observed {TARGET} trend — sensor {sensor}')
plt.ylabel(TARGET)
plt.tight_layout()

In [ ]:
weekday = data.groupby(data.index.dayofweek).mean().mean(axis=1)
weekday.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekday.plot(kind='bar', title=f'Mean {TARGET} by day of week', ylabel=TARGET)
plt.tight_layout()

In [ ]:
recent = data.tail(2 * 24 * 60 // 5)[sample_sensors]
sns.heatmap(recent.T, cmap='viridis', cbar_kws={'label': TARGET})
plt.title(f'Recent {TARGET} by sensor and timestamp')
plt.xlabel('Recent observations')
plt.ylabel('Sensor')
plt.tight_layout()

In [ ]:
if coordinates is not None and {'sensor_id', 'latitude', 'longitude'}.issubset(coordinates.columns):
    latest = data.iloc[-1].rename(TARGET).rename_axis('sensor_id').reset_index()
    locations = coordinates.merge(latest, on='sensor_id', how='inner')
    plt.figure(figsize=(8, 6))
    points = plt.scatter(locations['longitude'], locations['latitude'], c=locations[TARGET], cmap='viridis')
    plt.colorbar(points, label=f'Latest observed {TARGET}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Sensor locations colored by latest observed value')
    plt.tight_layout()
else:
    print('No sensor GPS coordinates were supplied; geographic visualization is unavailable.')